In [13]:
import pandas as pd
import numpy as np
from collections import Counter
from collections import defaultdict

#custom functions
import heuristic_functions

In [14]:
%run initialize_heuristic_data.py

In [15]:
%who


A	 A_p	 A_v	 Counter	 P	 P_a	 P_v	 V	 V_a	 
V_p	 all_antigens	 antigen_counts	 antigens	 capacity_data	 capacity_path	 columns	 data	 defaultdict	 
delta	 demand_80	 demand_80_MCV	 demand_80_expanded	 demand_80_final	 demand_path	 f_start	 file_path	 fulfill_demand	 
heuristic_functions	 i_start	 interim_demand_DF	 inventory_DF	 inventory_DF_MCV	 missed_doses	 missed_doses_MCV	 money_path	 np	 
pd	 price_data	 price_list	 ratio_DF	 ratio_DF_MCV	 s_start	 sheet_names	 tender_schedules	 tender_schedules_MCV	 
vaccine_consumption_percent	 vaccine_purchases	 years	 


In [16]:
# Selecting only the rows for 'Measles', 'Mumps', and 'Rubella' in both datasets
demand_80_MCV = demand_80_final[demand_80_final['antigen'].isin(['Measles', 'Mumps', 'Rubella'])]
interim_demand_DF = demand_80_MCV.copy()
tender_schedules_MCV = tender_schedules[tender_schedules['Antigen'].isin(['Measles', 'Mumps', 'Rubella'])]
# i_start_MCV = i_start[i_start['Vaccine'].isin(['M', 'MR', 'MMR'])]
missed_doses_MCV = missed_doses[missed_doses['Antigen'].isin(['Measles', 'Mumps', 'Rubella'])]
ratio_DF_MCV = pd.DataFrame()
inventory_DF_MCV = inventory_DF[inventory_DF['Vaccine'].isin(['M', 'MR', 'MMR'])]

#logic to setup least covered antigens:
# Flatten the list of all antigens from all vaccines
all_antigens = [antigen for antigens in A_v.values() for antigen in antigens]
# Count the occurrences of each antigen
antigen_counts = Counter(all_antigens)
# least_covered_antigens = sorted(antigen_counts.keys(), key=lambda x: antigen_counts[x], reverse=False)

In [17]:
def fulfill_demand(vaccine_price_list, antigen, interim_demand, manufacturer_capacities_df, inventory_DF, year, total_price): #add total price to track
    # Create a copy of the demand DataFrame to work with
    # interim_demand = interim_demand_DF

    # Check if the given year exists in the DataFrame
    if year not in manufacturer_capacities_df.columns:
        raise KeyError(f"Year {year} is not a valid column in the manufacturer capacities DataFrame.")
    
    # Sort the list by price to ensure we start with the lowest price - this is a techincal re-sort, since the initial list is sorted alredy
    vaccine_price_list.sort(key=lambda x: x['Price'])
    
    # Initialize variables
    total_demand_filled = 0
    demand_unfilled = []
    # message = ["None"]
    inventory_used = []

    for entry in vaccine_price_list: 
        print(f"ENTRY: {entry['Vaccine']}")
        manufacturer = entry['Manufacturer']
        vaccine = entry['Vaccine']
        price = entry['Price']

        manufacturer_row = manufacturer_capacities_df[manufacturer_capacities_df['Manufacturer'] == manufacturer]
        if not manufacturer_row.empty:
            capacity = manufacturer_row.iloc[0][year+1]
            print(f"Current capacity: {capacity} for {manufacturer}")
        else:
            raise ValueError(f"Manufacturer '{manufacturer}' not found in the list of manufacturers.")
        
        # Check the demand for the current vaccine in interim_demand
        if antigen in interim_demand.iloc[:, 0].tolist():
            antigen_demand = interim_demand.loc[interim_demand['antigen'] == antigen, year + 1].iloc[0]
            print(f"antigen {antigen} demand: {antigen_demand}")
        else:
            raise ValueError(f"Antigen '{antigen}' not found in the list of antigens.")
        
        if antigen_demand == 0: #stop checking if demand is satisfied
            break
        

        if capacity > 0 and antigen_demand >0:
            print("Winning")
            pairs = {'antigen_demand': antigen_demand, 'producer_capacity': capacity}
            print(pairs)
            min_key = min(pairs, key=pairs.get)
            amount_to_fill = pairs[min_key]
            print(f"Amount to fill: {amount_to_fill}, from {min_key}")
            #update interim_demand for antigen
            print(f"previous interim {antigen} demand: {interim_demand.loc[interim_demand['antigen'] == antigen, year+1].iloc[0]}")
            interim_demand.loc[interim_demand['antigen'] == antigen, year+1] -= amount_to_fill
            print(f"updated interim {antigen} demand: {interim_demand.loc[interim_demand['antigen'] == antigen, year+1].iloc[0]}")
            print("Updating other antigen future demand for year + 1")
            antigens = A_v.get(vaccine, [])
            if antigen in antigens:
                antigens.remove(antigen)
            else:
                continue
            print(f"other antigens: {antigens}")
            for ant in antigens:
                interim_demand.loc[interim_demand['antigen'] == ant, year+1] -= amount_to_fill
                print(f"updating demand for {ant}")
                print(f"updated interim {ant} demand: {interim_demand.loc[interim_demand['antigen'] == ant, year+1].iloc[0]}")
            dict = {manufacturer: amount_to_fill}
            inventory_used.append(dict) #used for reporting and charting post run
            total_demand_filled += amount_to_fill
            total_price += amount_to_fill * price
            #inventory year + 1 + AMOUNT_TO_FILL
            inventory_DF.loc[inventory_DF['Vaccine']==vaccine, 'Amount'] += amount_to_fill
            #capacity year + 1 - amount_to_fill
            manufacturer_capacities_df.loc[manufacturer_capacities_df['Manufacturer']==manufacturer, year + 1] -= amount_to_fill
            # message = ["Demand fully fulfilled" if interim_demand.loc[interim_demand['antigen'] == antigen, year+1].iloc[0] == 0 else f"Demand not fully fulfilled"]

    #need to return the following items here:
    #I need to return the interim_demand somehow to update the interim_demand_DF outside this function
    return {
    'Inventory_Used': inventory_used,
    'Demand_Filled': total_demand_filled,
    'Demand_unfilled': demand_unfilled,
    'Total_Price': total_price,
    # 'Details': details,
    # 'Message': message
    }


        


In [18]:
inventory_DF_MCV

,Vaccine,Amount
4,M,257581400.0
5,MR,837500100.0
6,MMR,78464000.0


In [19]:
capacity_data.loc[capacity_data['Manufacturer']=='Serum_Institute']

,Manufacturer,1,2,3,4,5,6,7,8,9,10
14,Serum_Institute,587114096,587114096,587114096,587114096,587114096,587114096,587114096,587114096,587114096,587114096


In [20]:
price_list = heuristic_functions.get_manufacturer_vaccine_price('Measles', price_data, V_a, P_v, 4)

result = fulfill_demand(price_list, 'Measles', interim_demand_DF, capacity_data, inventory_DF_MCV, 4)
# print(result)

ENTRY: M
Current capacity: 45911731 for PT_Bio
antigen Measles demand: 377569100
Winning
{'antigen_demand': 377569100, 'producer_capacity': 45911731}
Amount to fill: 45911731, from producer_capacity
previous interim Measles demand: 377569100
updated interim Measles demand: 331657369
Updating other antigen future demand for year + 1
other antigens: []
ENTRY: M
Current capacity: 587114096 for Serum_Institute
antigen Measles demand: 331657369
Winning
{'antigen_demand': 331657369, 'producer_capacity': 587114096}
Amount to fill: 331657369, from antigen_demand
previous interim Measles demand: 331657369
updated interim Measles demand: 0
Updating other antigen future demand for year + 1
ENTRY: MR
Current capacity: 587114096 for Serum_Institute
antigen Measles demand: 0


In [21]:
result

{'Inventory_Used': [{'PT_Bio': 45911731}],
 'Demand_Filled': 45911731,
 'Demand_unfilled': [],
 'Total_Price': 12166608.715,
 'Message': ['Demand not fully fulfilled']}